# Ideas planner, second list, idea 4: an answer path without LoFTR, by agreement of part correlation peaks
At each shortlisted place (the 10 best coarse peaks of each of the six standard queries) slide the four quadrants of the canvas separately over +-3 cells and note each part's best zncc and where it peaks. A right place should have all four parts peaking at about the same shift with a high value; a look-alike gives parts that disagree. Features: the lowest part value, the mean part value, the spread of the four part offsets (cells) and the coarse score of the whole canvas.
Reading: per candidate, label = within 1 sigma of the truth; AUC of each feature (and of the coarse score alone) over all candidates; per photo, whether the candidate with the best feature is right, and the lowest-error false best.
Falsification (planned): dead if no threshold gives 0 wrong on the NUDT draw while rescuing at least 4 of the 12 low-altitude official skips. Run on official150 (TAG official150) and on a NUDT draw (GATE_DATA=nudt NUDT_SPLIT=valid TAG=nudt_valid1212 POOL_PHOTOS=1425).

In [1]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

official official150 pool 500 limit 0


In [2]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = replay.load_set(T7 / "rich_dump" / TAG)

In [3]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [1.0, 1.05])
from train.checks import part_maps
test_set = pipeline.test_set
photos = list(dump)
if N_LIMIT:
    photos = photos[:N_LIMIT]
masks, _ = part_maps.part_masks(224 // 4)
print(len(photos), "photos")

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 111 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 125 s
150 photos


In [4]:
rows = []
start = time.time()
for pid in photos:
    item = test_set.items[pid]; level = dump[pid]["level"]; truth = dump[pid]["truth"]; sigma = dump[pid]["sigma"]; height = dump[pid]["height"]
    decision = replay.decide_current(dump[pid])
    for q in replay.query_ids(dump[pid], STANDARD):
        query = dump[pid]["queries"][q]
        canvas, valid = render_query(item, pipeline.camera, level, query_geometry(query["heading"], query["scale"]))
        for k, peak in enumerate(query["peaks"][:10]):
            if peak[1] > 1e8:
                continue
            values, offsets = [], []
            for mask in masks.values():
                value, dr, dc = part_maps.local_peak(pipeline.store, canvas, valid, level, peak[1], peak[2], 3, mask)
                values.append(value); offsets.append((dr, dc))
            offsets = np.array(offsets)
            spread = max(np.hypot(*(offsets[i] - offsets[j])) for i in range(4) for j in range(i + 1, 4))
            whole, _, _ = part_maps.local_peak(pipeline.store, canvas, valid, level, peak[1], peak[2], 3, None)
            error = np.hypot(peak[1] - truth[0], peak[2] - truth[1]) / sigma
            rows.append(dict(photo=pid, height=height, ours_stage=decision["stage"], ours_err=decision["err_sigma"], query=q, k=k, coarse=float(peak[0]),
                             part_min=min(values), part_mean=float(np.mean(values)), spread=float(spread), whole=whole, inliers=float(query["inliers"][k]),
                             err_sigma=float(error), right=int(error <= 1.0)))
    if len(rows) % 200 < 60:
        print(pid, "%.0f s" % (time.time() - start), flush=True)
    torch.cuda.empty_cache()
F = pd.DataFrame(rows)
F.to_csv(T7 / ("idea4_part_peaks_%s.csv" % TAG), index=False)
print(len(F), "candidate rows;", int(F.right.sum()), "right ones")

5 5 s


8 9 s


11 13 s


23 18 s


30 22 s


35 26 s


... (36 outputs left out)

360 187 s


369 191 s


376 195 s


9000 candidate rows; 474 right ones


In [5]:
from sklearn.metrics import roc_auc_score
F["part_score"] = F.part_min - 0.05 * F.spread
out = []
for feature in ["coarse", "whole", "part_min", "part_mean", "part_score", "inliers"]:
    out.append(dict(feature=feature, auc=round(roc_auc_score(F.right, F[feature]), 3)))
out.append(dict(feature="spread (lower = right)", auc=round(roc_auc_score(F.right, -F.spread), 3)))
print(pd.DataFrame(out).to_string(index=False))
best = F.loc[F.groupby("photo")["part_score"].idxmax()]
best_inliers = F.loc[F.groupby("photo")["inliers"].idxmax()]
print("photos where the best part_score candidate is right: %d of %d; best LoFTR-inlier candidate right: %d" % (best.right.sum(), len(best), best_inliers.right.sum()))
skipped = F[F.ours_stage == "skip"].photo.unique()
print("photos ours skips:", len(skipped), " of which the part_score pick is right:", int(best[best.photo.isin(skipped)].right.sum()))
rows = []
for threshold in np.arange(0.3, 1.01, 0.05):
    answered = best[best.part_score >= threshold]
    rows.append(dict(threshold=round(threshold, 2), answered=len(answered), right=int(answered.right.sum()), wrong=int((answered.err_sigma > 8).sum()),
                     rescues=int(answered[answered.photo.isin(skipped)].right.sum())))
T = pd.DataFrame(rows); T.to_csv(T7 / ("idea4_threshold_%s.csv" % TAG), index=False); print(T.to_string(index=False))

               feature   auc
                coarse 0.872
                 whole 0.872
              part_min 0.976
             part_mean 0.954
            part_score 0.894
               inliers 0.946
spread (lower = right) 0.818
photos where the best part_score candidate is right: 100 of 150; best LoFTR-inlier candidate right: 119
photos ours skips: 16  of which the part_score pick is right: 12
 threshold  answered  right  wrong  rescues
      0.30         0      0      0        0
      0.35         0      0      0        0
      0.40         0      0      0        0
      0.45         0      0      0        0
      0.50         0      0      0        0
      0.55         0      0      0        0
      0.60         0      0      0        0
      0.65         0      0      0        0
      0.70         0      0      0        0
      0.75         0      0      0        0
      0.80         0      0      0        0
      0.85         0      0      0        0
      0.90         0      0